In [4]:
import sys
sys.path.insert(0, "..")

from optimal_long_short.model_params import KouParams
from optimal_long_short.market_params import MarketParams
from optimal_long_short.strategy import UnitExposureLongShortStrategy
from optimal_long_short.inversion import TalbotInverter, StehfestInverter, DeHoogInverter
from optimal_long_short.moments import ConditionalMoments

In [7]:
# --- model and strategy parameters ---
params = KouParams(
    mu1=-0.05, sigma1=0.20, lam1=1.0, p1=0.5, eta1_pos=1/10.1, eta1_neg=1/10.0,
    mu2=0.00, sigma2=0.15, lam2=1.0, p2=0.5, eta2_pos=1/10.1, eta2_neg=1/10.0,
    rho=0.3,
)
market = MarketParams(b=0.8, S10=1.0, S20=1.0)
strategy = UnitExposureLongShortStrategy(h0=0.5, market=market, T=1)

print(f"h0 = {strategy.h0},  T = {strategy.T}")
print(f"w1 = {strategy.w1:.6f},  w2 = {strategy.w2:.6f}")
print(f"b  = {market.b}")

h0 = 0.5,  T = 1
w1 = 1.942594,  w2 = 0.942594
b  = 0.8


In [8]:
# --- compare three inverters ---
inverters = {
    "Talbot (M=32)":   TalbotInverter(M=32),
    "Stehfest (N=12)": StehfestInverter(N=12),
    "de Hoog (M=24)":  DeHoogInverter(M=24),
}

print(f"{'Inverter':<20} {'p_surv':>10} {'E[Pi|surv]':>14} {'Var(Pi|surv)':>16}")
print("-" * 64)
for name, inv in inverters.items():
    cm = ConditionalMoments(params=params, strategy=strategy, inverter=inv)
    ps  = cm.p_surv()
    mu  = cm.conditional_mean()
    var = cm.conditional_variance()
    print(f"{name:<20} {ps:>10.6f} {mu:>14.6f} {var:>16.6f}")

Inverter                 p_surv     E[Pi|surv]     Var(Pi|surv)
----------------------------------------------------------------
Talbot (M=32)          0.900636       0.980780         0.183774
Stehfest (N=12)        0.900558       0.980841         0.183750
de Hoog (M=24)       625.069915       0.898706         0.279531


/Users/francis/Defi_interestRate/notebooks/../optimal_long_short/inversion.py:287: RuntimeWarning: invalid value encountered in divide
  q[:-2, r] = q[1:-1, r - 1] * e[1:-1, r] / e[:-2, r]


In [9]:
# --- sensitivity to h0 (Talbot) ---
import numpy as np

h0_grid = np.linspace(0.1, 2.0, 20)
inv = TalbotInverter(M=32)

p_survs, means, variances = [], [], []
for h0 in h0_grid:
    strat = UnitExposureLongShortStrategy(h0=float(h0), market=market, T=1.0)
    cm = ConditionalMoments(params=params, strategy=strat, inverter=inv)
    p_survs.append(cm.p_surv())
    means.append(cm.conditional_mean())
    variances.append(cm.conditional_variance())

print(f"{'h0':>6} {'p_surv':>10} {'E[Pi|surv]':>14} {'Var(Pi|surv)':>16}")
print("-" * 50)
for h0, ps, mu, var in zip(h0_grid, p_survs, means, variances):
    print(f"{h0:>6.2f} {ps:>10.6f} {mu:>14.6f} {var:>16.6f}")

    h0     p_surv     E[Pi|surv]     Var(Pi|surv)
--------------------------------------------------
  0.10   0.263796       1.709104         0.576983
  0.20   0.497635       1.337140         0.356155
  0.30   0.685115       1.142331         0.262291
  0.40   0.817467       1.036907         0.213729
  0.50   0.900636       0.980780         0.183774
  0.60   0.948286       0.952379         0.162326
  0.70   0.973873       0.939189         0.145436
  0.80   0.987061       0.933980         0.131563
  0.90   0.993688       0.932741         0.120034
  1.00   0.996960       0.933347         0.110451
  1.10   0.998552       0.934715         0.102500
  1.20   0.999318       0.936314         0.095908
  1.30   0.999681       0.937899         0.090434
  1.40   0.999852       0.939369         0.085871
  1.50   0.999932       0.940690         0.082049
  1.60   0.999969       0.941861         0.078830
  1.70   0.999986       0.942894         0.076101
  1.80   0.999994       0.943803         0.073774